# Visual Question Answering (VQA) End-to-End Pipeline

This notebook demonstrates an end-to-end VQA pipeline using Vision-Language Models (BLIP):
1. Loading a VQA dataset subset (`HuggingFaceM4/VQAv2`) using `src.data_loader`.
2. Performing Exploratory Data Analysis (EDA) visualizing sample images, questions, and ground-truth human answers.
3. Running batch inference with `src.model.VQAClassifier`.
4. Evaluating model performance using Exact Match (EM) accuracy.

In [ ]:
import sys
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Ensure src directory is on python path
sys.path.append(os.path.abspath('../src'))

from data_loader import load_vqa_subset, run_eda, VQADataset, get_vqa_dataloader
from model import VQAClassifier

## 1. Load VQA Dataset & Exploratory Data Analysis (EDA)

In [ ]:
# Load validation subset of VQAv2 dataset (50 samples for quick evaluation)
vqa_data = load_vqa_subset(split='validation[:50]')
print(f"Loaded {len(vqa_data)} VQA samples.")

# Visualize 3 sample images with questions and top human answers
run_eda(vqa_data, num_samples=3)

## 2. Model Initialization & Batch Inference

In [ ]:
# Initialize VQAClassifier model (automatically maps to CUDA GPU if available)
model = VQAClassifier(model_name="Salesforce/blip-vqa-base")

# Extract images and questions
images = [sample['image'] for sample in vqa_data]
questions = [sample['question'] for sample in vqa_data]

# Run batch inference in chunks of 8
batch_size = 8
predictions = []

for i in tqdm(range(0, len(images), batch_size), desc="Running Batch Inference"):
    batch_imgs = images[i:i + batch_size]
    batch_qs = questions[i:i + batch_size]
    batch_preds = model.batch_predict(batch_imgs, batch_qs)
    predictions.extend(batch_preds)

print(f"Generated {len(predictions)} predictions.")

## 3. Compute Exact Match (EM) Accuracy

In [ ]:
# Extract ground truth human answers
ground_truths = []
for sample in vqa_data:
    if "multiple_choice_answer" in sample and sample["multiple_choice_answer"]:
        gt = sample["multiple_choice_answer"].strip().lower()
    elif "answers" in sample and sample["answers"]:
        first_ans = sample["answers"][0]
        gt = (first_ans.get("answer", "") if isinstance(first_ans, dict) else str(first_ans)).strip().lower()
    else:
        gt = sample.get("answer", "").strip().lower()
    ground_truths.append(gt)

# Compare predictions with ground truth answers
print("Sample Predictions vs Ground Truth:")
print("-" * 65)
for i in range(min(10, len(predictions))):
    pred = predictions[i].strip().lower()
    gt = ground_truths[i]
    is_match = (pred == gt)
    print(f"Q: {questions[i]}")
    print(f"   Predicted: {pred:<15} | Ground Truth: {gt:<15} | Match: {is_match}")

# Overall Exact Match Accuracy across all samples
total_correct = sum(1 for pred, gt in zip(predictions, ground_truths) if pred.strip().lower() == gt)
accuracy = (total_correct / len(ground_truths)) * 100

print("=" * 65)
print(f"Total Samples Evaluated: {len(ground_truths)}")
print(f"Exact Matches: {total_correct}")
print(f"Exact Match Accuracy: {accuracy:.2f}%")